# Emotion Extractor Comparison — Colab Notebook

**Purpose:** Run MT-DDAMFN, HSEmotion, and EmoNet on the 68 interactions that have Imitator ground-truth, then rank them against Imitator on valence/arousal CCC + 8-class agreement. The winner will extract features for all 443 interactions.

**Runtime expectations (Colab Pro, T4 High-RAM baseline — scales with VRAM via `BATCH_MULT`):**
- First-time Drive mount + clone both repos into Drive: ~8-15 min (clones go to Drive so they persist across sessions — subsequent runs skip this)
- Subsequent-run Drive mount + `git pull`: ~1-2 min
- Install deps: ~2-3 min
- Verify 136 Imitator emotion_features.npz are present (already committed to the repo): ~10 sec
- Download 136 video + boxes NPZ from S3 (for inference): ~30-45 min
- MT-DDAMFN on 136 files: ~45-60 min
- HSEmotion on 136 files: ~30-45 min
- EmoNet on 136 files: ~90-120 min
- Evaluation + plots: ~5 min
- Total comparison: ~4-5 hours, ~10-12 compute units

**If Colab allocates you a bigger GPU than T4** (L4, A100, H100, RTX PRO 6000 Blackwell, etc.), Cell 4 auto-detects VRAM and sets `BATCH_MULT = vram_gb // 16` (capped at 8×). Extraction wall-clock drops roughly linearly with batch size on these small CNNs — e.g., a 96 GB Blackwell (BATCH_MULT=6) should finish extraction in ~40-60 min instead of ~3-4 hours. The S3 download stage (~30-45 min) is network-bound and unaffected.



**Prerequisite — set up a Colab Secret named `GITHUB_PAT`** (one-time, per Colab account). See the markdown above Cell 1 for step-by-step. Without this, Cell 1 will fail with a clear instruction error. The PAT never lives in this notebook file.

**Caveats (read before running):**
1. Imitator `emotion_scores` are raw logits, not probabilities. Direct numerical comparison requires affine calibration; argmax agreement is fine.
2. Imitator's 8-class label ordering is not documented — one of the eval cells builds a class-index confusion matrix to detect reshuffling before computing F1.
3. Imitator's valence/arousal are on [-1, +1] but the empirical range in our sample is narrower (~[-0.5, +0.9]). External models also output [-1, +1]. CCC is scale-sensitive so we report raw CCC AND CCC-after-affine-recalibration.
4. MT-DDAMFN loading code below is best-effort; you may need to tweak the state_dict key prefix after inspection.


**Resuming after a runtime restart or session disconnect.** Colab wipes the VM (pip installs, `/content/` files, Python state) on every reconnect, but Drive contents — the repo clones and anything under `/content/drive/...` — survive. To resume: re-run Cells 1-4 in order (the `GITHUB_PAT` Colab Secret is account-level, so it persists across restarts — no need to re-enter it) (each is idempotent — `git pull` on already-cloned repos, skip-if-installed pip, etc.), then jump back to whichever cell you were on. Cells 5, 6, 9, 10, and 11 are all resume-safe: they skip files already on disk, and Cells 9/10/11 additionally validate that existing NPZs can be opened before skipping (a corrupt file from a killed prior run gets re-extracted rather than silently accepted).


## Runtime selection

Before running anything, connect Colab to the right runtime:

1. Click the `Connect` dropdown (top-right).
2. Change runtime type: **GPU → T4**, **Shape → High-RAM**.
3. If `L4` happens to be available with no queue, it's ~2× faster than T4 at ~2× the cost — acceptable swap. Avoid V100 and A100 (burn budget with little speedup on small CNNs).

Expected allocation: T4 GPU + 25 GB system RAM + 150 GB local disk. Compute-unit cost: ~2.4 units/hour.


## Cell 1 — Mount Drive and clone both repos into `535_project_data/repos/`

Both repos live in Google Drive at `MyDrive/535_project_data/repos/` so they survive Colab session disconnects. On first run this block does the initial clones (~8-15 min because Drive is slow for lots of small files). On subsequent runs it just mounts + pulls.

**Prerequisite — one-time Colab Secret setup (per Colab account):**
1. In Colab, click the **key icon** in the far-left sidebar to open the **Secrets** panel.
2. Click **+ Add new secret**.
3. **Name:** `GITHUB_PAT` (exact case).
4. **Value:** paste your GitHub fine-grained PAT with read (and write, if you plan to `git push` from Colab) access to `USC-CSCI-535-SP26/csci535-project`.
5. Toggle **Notebook access** ON for this notebook.

Once the secret is set, the notebook loads it at runtime via `userdata.get('GITHUB_PAT')` — the PAT never lives in the .ipynb file, never hits `.git/config`, and is erased automatically when the runtime dies.


In [ ]:
# --- Load PAT from Colab Secrets (one-time setup described in the markdown above) ---
from google.colab import userdata
try:
    GH_PAT = userdata.get('GITHUB_PAT')
except userdata.SecretNotFoundError:
    raise RuntimeError(
        "Colab Secret 'GITHUB_PAT' is not set. Open the Secrets panel "
        "(key icon, far-left sidebar), click '+ Add new secret', name it "
        "'GITHUB_PAT', paste your fine-grained PAT, and toggle 'Notebook "
        "access' ON for this notebook."
    )
except userdata.NotebookAccessError:
    raise RuntimeError(
        "Colab Secret 'GITHUB_PAT' exists but this notebook doesn't have "
        "access. Open the Secrets panel and toggle 'Notebook access' ON "
        "for this notebook."
    )
GH_USER = "rasikaramanan"

# --- Mount Drive ---
from google.colab import drive
drive.mount('/content/drive')

import os, re
from pathlib import Path

REPOS = Path('/content/drive/MyDrive/535_project_data/repos')
REPOS.mkdir(parents=True, exist_ok=True)

CSCI535_REPO  = REPOS / 'csci535-project'
SEAMLESS_REPO = REPOS / 'seamless_interaction'

# --- Configure git to read the PAT from an env var via GIT_ASKPASS ---
# This keeps the PAT OUT of .git/config (otherwise it would live on Drive forever)
# while still allowing `git pull` / `git push` to work throughout this session.
# When the runtime dies, the env var and /tmp/git_askpass.sh are both gone.
os.environ['GITHUB_PAT'] = GH_PAT
_askpass = Path('/tmp/git_askpass.sh')
_askpass.write_text('#!/bin/bash\necho "$GITHUB_PAT"\n')
_askpass.chmod(0o755)
os.environ['GIT_ASKPASS'] = str(_askpass)
os.environ['GIT_TERMINAL_PROMPT'] = '0'

def _scrub_embedded_pat(repo_path: Path):
    """If a prior (pre-secret) session embedded the PAT in .git/config, rewrite to a clean URL."""
    cfg = repo_path / '.git' / 'config'
    if not cfg.exists():
        return
    text = cfg.read_text()
    if re.search(r'https://[^@/\s]+:[^@\s]+@github\.com', text):
        cleaned = re.sub(
            r'https://[^@/\s]+:[^@\s]+@github\.com',
            f'https://{GH_USER}@github.com',
            text,
        )
        cfg.write_text(cleaned)
        print(f"Scrubbed embedded PAT from {cfg}")

# --- Clone csci535-project (private) if not already on Drive, else pull latest ---
if not (CSCI535_REPO / '.git').exists():
    # No PAT in the URL — git calls GIT_ASKPASS when it needs the password.
    !git clone https://{GH_USER}@github.com/USC-CSCI-535-SP26/csci535-project.git "{CSCI535_REPO}"
else:
    _scrub_embedded_pat(CSCI535_REPO)
    print(f"csci535-project already cloned at {CSCI535_REPO}; pulling...")
    !cd "{CSCI535_REPO}" && git pull

# --- Clone seamless_interaction (public — no auth needed) ---
if not (SEAMLESS_REPO / '.git').exists():
    !git clone https://github.com/facebookresearch/seamless_interaction.git "{SEAMLESS_REPO}"
else:
    print(f"seamless_interaction already cloned at {SEAMLESS_REPO}; pulling...")
    !cd "{SEAMLESS_REPO}" && git pull

# --- Drop the PAT from the Python namespace (os.environ copy kept for git ops this session) ---
del GH_PAT

# --- Verify both trees landed ---
!ls "{CSCI535_REPO}/scripts/valence_arousal/download_emotion_features.py"
!ls "{SEAMLESS_REPO}/assets/filelist.csv"


## Cell 2 — Install all Python dependencies


In [ ]:
# Precondition: Cell 1 must have defined SEAMLESS_REPO and CSCI535_REPO.
assert 'SEAMLESS_REPO' in globals() and 'CSCI535_REPO' in globals(), \
    "Run Cell 1 first (it mounts Drive and defines the repo paths)."

# Seamless Interaction package (for S3 downloads of video + boxes) — installed from the Drive clone
!cd "{SEAMLESS_REPO}" && pip install -q -e . --no-deps
# wget is imported at module load in seamless_interaction/fs.py and does the actual S3 downloads.
# huggingface_hub is used by fs.py for filelist.csv fallback lookups.
!pip install -q wget huggingface_hub boto3

# Core deps
!pip install -q torch torchvision opencv-python-headless timm einops
!pip install -q pandas numpy scipy scikit-learn matplotlib seaborn tqdm

# HSEmotion (enet_b0_8_va_mtl)
!pip install -q hsemotion hsemotion-onnx

# EmoNet + MT-DDAMFN support repos — these are cloned to VM-local /content (ephemeral is fine;
# they're small and just supply source + weights, not data we need to persist).
!git clone -q https://github.com/face-analysis/emonet /content/emonet
!git clone -q https://github.com/av-savchenko/face-emotion-recognition /content/fer_savchenko
!git clone -q https://github.com/SainingZhang/DDAMFN /content/ddamfn

# Verify
!nvidia-smi --query-gpu=name,memory.total --format=csv
!ls /content/emonet/pretrained/ | head
!ls /content/fer_savchenko/src/ABAW/ABAW6/ 2>/dev/null | head




Note: Cell 2 re-uses the `SEAMLESS_REPO` variable from Cell 1, so Cell 1 must run first.

**Expected dependency-resolver warnings (safe to ignore):** installing `seamless-interaction` with `--no-deps` skips its entire pyproject requirement list, which makes pip complain about missing `jupyterlab`, `mypy`, `pre-commit`, `pylance`, `types-requests`, `streamlit`, `s3fs`, `webdataset`, `plotly`. None of these are imported on the S3-download code path — only `wget` and `huggingface_hub` are, and Cell 2 installs both explicitly. You will also see `pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.` — this is a pip warning, not an error; the install succeeded.


## Cell 3 — Verify Imitator ground truth NPZs came through the clone

The 136 `emotion_features.npz` files are **committed to the csci535-project repo** (~42 MB total across both_annotated_interactions/ and single_annotated_interactions/). They arrived via the Cell 1 clone — no S3 re-download needed. This cell just confirms the count.


In [ ]:
from pathlib import Path

npz_paths = sorted(
    list((CSCI535_REPO / 'both_annotated_interactions').rglob('emotion_features.npz')) +
    list((CSCI535_REPO / 'single_annotated_interactions').rglob('emotion_features.npz'))
)
print(f"Found {len(npz_paths)} emotion_features.npz files (expected: 136)")

if len(npz_paths) < 136:
    print(f"\nWARNING: fewer than 136 NPZs — found {len(npz_paths)} / 136.")
    print("The most common cause is that the missing NPZs were never pushed to GitHub from your local machine:")
    print("  1. On your Mac, `cd` into the local csci535-project checkout.")
    print("  2. Run `git status` — if emotion_features.npz files appear as untracked/unstaged/unpushed, `git add`, `git commit`, `git push`.")
    print("  3. Back in Colab, run `!cd \"{CSCI535_REPO}\" && git pull` to refresh the Drive clone.")
    print("\nIf you know they ARE pushed, then the Drive FUSE clone may have dropped files:")
    print("  - Delete the csci535-project folder in Drive (web UI or Finder), then re-run Cell 1.")
    print("Alternative (slower, but guaranteed): re-download missing NPZs directly from S3:")
    print(f'  !python "{CSCI535_REPO}/scripts/valence_arousal/download_emotion_features.py" \\')
    print(f'      --repo-path "{SEAMLESS_REPO}" \\')
    print(f'      --interactions-dir "{CSCI535_REPO}/both_annotated_interactions"')
    print(f'  !python "{CSCI535_REPO}/scripts/valence_arousal/download_emotion_features.py" \\')
    print(f'      --repo-path "{SEAMLESS_REPO}" \\')
    print(f'      --interactions-dir "{CSCI535_REPO}/single_annotated_interactions"')
else:
    print("OK — Imitator ground truth is ready to use.")


## Cell 4 — Identify the 136 participant-files and set up paths


In [ ]:
import sys
import pandas as pd
import torch
from pathlib import Path

# Make the seamless_interaction package importable regardless of whether `pip install -e`
# in Cell 2 actually registered. Editable installs where the source lives on a FUSE-mounted
# Google Drive fail silently often enough that we add this belt-and-suspenders path insert.
SEAMLESS_SRC = str(SEAMLESS_REPO / 'src')
if SEAMLESS_SRC not in sys.path:
    sys.path.insert(0, SEAMLESS_SRC)

PROJECT = CSCI535_REPO   # from Cell 1: /content/drive/MyDrive/535_project_data/repos/csci535-project
DATA_DIR = '/content/seamless_data'  # local Colab VM storage for video + boxes NPZs
Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

# Optional: persist predictions to Drive so you don't lose them on disconnect
# Uncomment if desired:
# WORK = '/content/drive/MyDrive/535_project_data/arousal_valence_comparison'
WORK = '/content/arousal_valence_comparison'   # VM-local; survives within a session
Path(WORK).mkdir(parents=True, exist_ok=True)

# =========================================================================
# GPU sanity + batch-size scaling
# =========================================================================
# Colab can allocate anything from a T4 (16 GB) up to an RTX PRO 6000 Blackwell (96 GB)
# depending on the session. Batch sizes in Cells 9/10/11 are written as `64 * BATCH_MULT`
# so they adapt automatically. A small smoke test below also catches the case where
# PyTorch wasn't built with kernels for a brand-new GPU arch (symptom on Blackwell: you
# get "CUDA error: no kernel image is available for execution on the device" when you
# run your first model forward pass, not at import time).
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb  = torch.cuda.get_device_properties(0).total_memory / 1e9
    # Smoke test: run a real op on the GPU. If PyTorch lacks a kernel for this arch,
    # the matmul will raise here — better to fail now than 2 hours into extraction.
    try:
        _ = (torch.randn(1024, 1024, device='cuda') @ torch.randn(1024, 1024, device='cuda')).sum().item()
        cuda_ok = True
    except RuntimeError as e:
        cuda_ok = False
        print(f"ERROR: GPU smoke test failed: {e}")
        print("Likely fix: upgrade torch to a nightly/CUDA-12.4+ build, e.g.:")
        print("  !pip install --upgrade --pre torch torchvision --index-url https://download.pytorch.org/whl/nightly/cu124")
    # Scale batch sizes ~linearly with VRAM above the T4 baseline (16 GB), capped so we
    # don't blow up CPU-side crop staging memory.
    BATCH_MULT = max(1, min(8, int(vram_gb // 16)))
    print(f"GPU: {gpu_name}")
    print(f"VRAM: {vram_gb:.1f} GB   BATCH_MULT = {BATCH_MULT}x   cuda_ok = {cuda_ok}")
else:
    BATCH_MULT = 1
    print("WARNING: no CUDA GPU detected — extraction will fall back to CPU and be very slow.")

# Identify the 68 covered interactions from filelist.csv
filelist = pd.read_csv(SEAMLESS_REPO / 'assets' / 'filelist.csv')
filelist['interaction'] = filelist['file_id'].str.rsplit('_P', n=1).str[0]

OUR_443 = set()
for sub in ['both_annotated_interactions', 'single_annotated_interactions']:
    OUR_443.update(p.name for p in (PROJECT / sub).iterdir() if p.is_dir())

mask = filelist['interaction'].isin(OUR_443) & (filelist['has_imitator_movement'] == 1)
COVERED_ROWS = filelist.loc[mask].reset_index(drop=True)
COVERED_FILE_IDS = COVERED_ROWS['file_id'].tolist()
print(f"{len(COVERED_FILE_IDS)} participant-files across {COVERED_ROWS['interaction'].nunique()} interactions")
# Expected: 136 files across 68 interactions

# Label/split breakdown (needed for gather_file_id_data_from_s3)
print(COVERED_ROWS.groupby(['label','split']).size())


## Cell 5 — Sanity: download one video + boxes NPZ, inspect schema


In [ ]:
# Belt-and-suspenders: if Cell 4 hasn't run (e.g., runtime just restarted), re-apply the
# sys.path shim so this cell can import seamless_interaction on its own. Also protects against
# the ModuleNotFoundError you'd otherwise hit when `pip install -e` from a Drive FUSE path
# silently fails to register.
import sys
from pathlib import Path
_SEAMLESS_SRC = str(SEAMLESS_REPO / 'src')
if _SEAMLESS_SRC not in sys.path:
    sys.path.insert(0, _SEAMLESS_SRC)

from seamless_interaction.fs import SeamlessInteractionFS, DatasetConfig
import numpy as np

sample_row = COVERED_ROWS.iloc[0]
sample = sample_row['file_id']

# Resume-safe: skip the download if the sample's mp4 + npz are already on disk from a prior run.
sample_mp4 = next(Path(DATA_DIR).rglob(f'{sample}.mp4'), None)
sample_npz = next(Path(DATA_DIR).rglob(f'{sample}.npz'), None)
if sample_mp4 and sample_npz:
    print(f"Sample {sample} already on disk; skipping re-download.")
else:
    fs = SeamlessInteractionFS(
        config=DatasetConfig(
            label=sample_row['label'],
            split=sample_row['split'],
            local_dir=DATA_DIR,
            num_workers=4,
            preferred_vendors_only=False,
        )
    )
    fs.gather_file_id_data_from_s3(sample)

# Find downloaded files
for p in Path(DATA_DIR).rglob(f'{sample}.*'):
    print(p.suffix, f'{p.stat().st_size/1e6:.1f} MB', p)

# Inspect full NPZ schema (keypoints/boxes + movement features)
npz_path = next(Path(DATA_DIR).rglob(f'{sample}.npz'))
data = np.load(npz_path, allow_pickle=True)
print("\nNPZ keys:")
for k in data.files:
    arr = data[k]
    print(f"  {k:40s} shape={arr.shape} dtype={arr.dtype}")




Expected: a combined NPZ containing `boxes_and_keypoints:box`, `boxes_and_keypoints:is_valid_box`, `boxes_and_keypoints:keypoints`, and all `movement:*` features. Key names may use `:` or `/` as the separator — the helper cells below use `find_key()` which is tolerant.


## Cell 6 — Download video + boxes NPZ for all 136 files in parallel


In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def try_download(row, max_retries=2):
    fid, label, split = row['file_id'], row['label'], row['split']
    last_err = None
    for attempt in range(max_retries + 1):
        try:
            fs2 = SeamlessInteractionFS(
                config=DatasetConfig(label=label, split=split, local_dir=DATA_DIR,
                                     num_workers=2, preferred_vendors_only=False)
            )
            fs2.gather_file_id_data_from_s3(fid)
            return (fid, None)
        except Exception as e:
            last_err = f"{type(e).__name__}: {e}"
    return (fid, f"after {max_retries+1} attempts: {last_err}")

def _has_complete_bundle(fid):
    """A file is only 'done' if BOTH the mp4 and the combined NPZ are on disk.
    A partial download that left only one of the two would later silently break extraction."""
    mp4 = next(Path(DATA_DIR).rglob(f'{fid}.mp4'), None)
    npz = next(Path(DATA_DIR).rglob(f'{fid}.npz'), None)
    return mp4 is not None and npz is not None

existing = {fid for fid in COVERED_FILE_IDS if _has_complete_bundle(fid)}
todo_rows = [r for _, r in COVERED_ROWS.iterrows() if r['file_id'] not in existing]
print(f"Already have complete bundles: {len(existing)} / {len(COVERED_FILE_IDS)}")
print(f"Downloading: {len(todo_rows)}")

results = []
with ThreadPoolExecutor(max_workers=6) as ex:
    futures = [ex.submit(try_download, r) for r in todo_rows]
    for fut in tqdm(as_completed(futures), total=len(futures)):
        results.append(fut.result())

failures = [r for r in results if r[1]]
print(f"\nSucceeded: {len(results)-len(failures)}")
print(f"Failed: {len(failures)}")
for r in failures[:10]:
    print("  ", r)


## Cell 7 — Helpers: find NPZ keys, load video frames, crop faces


In [ ]:
import cv2, numpy as np, torch


# =========================================================================
# Extraction-output helpers (shared by Cells 9, 10, 11)
# =========================================================================
def save_atomic(path, **arrays):
    """Write an NPZ atomically. Prevents corrupt files when the runtime is
    killed mid-write — without this, the extraction loops' 'skip if file
    exists' check would accept a truncated NPZ as 'done'."""
    from pathlib import Path as _P
    path = _P(path)
    tmp = path.with_suffix(path.suffix + '.tmp')
    np.savez_compressed(tmp, **arrays)
    tmp.replace(path)  # POSIX-atomic rename

def is_valid_npz(path):
    """Return True iff `path` can be opened as an NPZ and its keys listed.
    Used by extraction cells to drop corrupt leftovers from a killed run."""
    try:
        with np.load(path) as d:
            _ = d.files
        return True
    except Exception:
        return False

def find_key(npz, needle):
    for k in npz.files:
        if needle in k:
            return k
    raise KeyError(needle)

def locate_files(fid, root=DATA_DIR):
    r = Path(root)
    return {
        'mp4': next(r.rglob(f'{fid}.mp4'), None),
        'npz': next(r.rglob(f'{fid}.npz'), None),
    }

def _to_xyxy(b, shape):
    h, w = shape[:2]
    b = np.asarray(b, dtype=np.float32)
    # If max <= 1.5, coords are normalized; else they're pixels
    if b.max() <= 1.5:
        b = b * np.array([w, h, w, h], dtype=np.float32)
    x1, y1, x2, y2 = b.tolist()
    if x2 < x1 or y2 < y1:
        cx, cy, bw, bh = b
        x1, y1, x2, y2 = cx - bw/2, cy - bh/2, cx + bw/2, cy + bh/2
    return x1, y1, x2, y2

def iter_face_crops(fid, crop_size=224, margin=0.2):
    """Yield (frame_idx, face_crop_rgb_uint8, is_valid) for every frame."""
    paths = locate_files(fid)
    if not paths['mp4'] or not paths['npz']:
        return
    data = np.load(paths['npz'], allow_pickle=True)
    try:
        box = data[find_key(data, 'box')]
        is_valid = data[find_key(data, 'is_valid_box')]
    except KeyError as e:
        print(f"{fid}: missing bbox key {e}")
        return

    cap = cv2.VideoCapture(str(paths['mp4']))
    n = 0
    while True:
        ok, bgr = cap.read()
        if not ok or n >= len(box):
            break
        valid = bool(is_valid[n]) if n < len(is_valid) else True
        x1, y1, x2, y2 = _to_xyxy(box[n], bgr.shape)
        w, h = x2 - x1, y2 - y1
        x1 = max(0, int(x1 - margin * w)); y1 = max(0, int(y1 - margin * h))
        x2 = min(bgr.shape[1], int(x2 + margin * w))
        y2 = min(bgr.shape[0], int(y2 + margin * h))
        if x2 - x1 < 10 or y2 - y1 < 10 or not valid:
            yield n, None, False
        else:
            crop = cv2.resize(bgr[y1:y2, x1:x2], (crop_size, crop_size))
            crop = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
            yield n, crop, True
        n += 1
    cap.release()

# Verify on one file (with a helpful hint if the sample isn't downloaded yet)
fid_test = COVERED_FILE_IDS[0]
_paths = locate_files(fid_test)
if not _paths['mp4'] or not _paths['npz']:
    print(f"Test file {fid_test} isn't downloaded yet. Run Cell 5 (or finish Cell 6) first, then re-run this cell.")
else:
    n_valid = sum(1 for i, crop, valid in iter_face_crops(fid_test) if valid and i < 100)
    print(f"First ~100 frames of {fid_test}: {n_valid} valid crops")


## Cell 8 — Load Imitator ground truth (from local emotion_features.npz)


In [ ]:
from pathlib import Path
import numpy as np

def find_imitator_npz(fid):
    """Locate emotion_features.npz for a given file_id, written by download_emotion_features.py."""
    iid, pid = fid.rsplit('_P', 1)
    pid = 'P' + pid
    for sub in ['both_annotated_interactions', 'single_annotated_interactions']:
        root = PROJECT / sub / iid
        if not root.exists(): continue
        for part_dir in root.iterdir():
            if part_dir.is_dir() and part_dir.name.endswith(pid):
                cand = part_dir / 'emotion_features.npz'
                if cand.exists():
                    return cand
    return None

def load_imitator_gt(fid):
    p = find_imitator_npz(fid)
    if p is None:
        return None
    d = np.load(p, allow_pickle=True)
    return {
        'valence': np.asarray(d['emotion_valence']).squeeze(),
        'arousal': np.asarray(d['emotion_arousal']).squeeze(),
        'scores':  np.asarray(d['emotion_scores']),
    }

# Verify
gt = load_imitator_gt(COVERED_FILE_IDS[0])
print({k: v.shape for k, v in gt.items()})
# Expected: valence (N,), arousal (N,), scores (N, 8)


## Cell 9 — MT-DDAMFN extraction


In [ ]:
import sys, torch, torch.nn.functional as F
sys.path.insert(0, '/content/ddamfn/DDAMFN++')
from networks.DDAM import DDAMNet

# Check weight file — filename may differ
!ls /content/fer_savchenko/src/ABAW/ABAW6/models/ 2>/dev/null
MT_DDAMFN_WEIGHTS = '/content/fer_savchenko/src/ABAW/ABAW6/models/mtl_ddamfn_abaw6.pth'  # EDIT IF DIFFERENT

device = 'cuda'

class MT_DDAMFN(torch.nn.Module):
    """Backbone + multi-task head: 8 emotion logits + valence + arousal."""
    def __init__(self, num_class=8):
        super().__init__()
        self.backbone = DDAMNet(num_class=num_class, num_head=2)
        feat_dim = self.backbone.fc.in_features if hasattr(self.backbone, 'fc') else 512
        self.va_head = torch.nn.Linear(feat_dim, 2)

    def forward(self, x):
        out, x_feats, *_ = self.backbone(x)
        va = torch.tanh(self.va_head(x_feats))
        return out, va[:, 0], va[:, 1]

model = MT_DDAMFN(num_class=8).to(device).eval()
sd = torch.load(MT_DDAMFN_WEIGHTS, map_location=device)
if 'state_dict' in sd: sd = sd['state_dict']
if 'model' in sd: sd = sd['model']
missing, unexpected = model.load_state_dict(sd, strict=False)
print(f"Missing keys: {len(missing)}, Unexpected: {len(unexpected)}")

IMG_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1,3,1,1)
IMG_STD  = torch.tensor([0.229, 0.224, 0.225], device=device).view(1,3,1,1)

@torch.no_grad()
def extract_mtddamfn(fid, batch_size=64*BATCH_MULT, crop_size=112):
    valid_mask = []
    batch_buf = []
    probs_all, val_all, aro_all = [], [], []

    def flush():
        x = torch.from_numpy(np.stack(batch_buf)).permute(0,3,1,2).float().to(device) / 255.
        x = (x - IMG_MEAN) / IMG_STD
        logits, v, a = model(x)
        probs_all.append(F.softmax(logits, dim=-1).cpu().numpy())
        val_all.append(v.cpu().numpy())
        aro_all.append(a.cpu().numpy())
        batch_buf.clear()

    for i, crop, valid in iter_face_crops(fid, crop_size=crop_size):
        valid_mask.append(valid)
        batch_buf.append(crop if valid else np.zeros((crop_size, crop_size, 3), dtype=np.uint8))
        if len(batch_buf) == batch_size:
            flush()
    if batch_buf: flush()

    return {
        'probs':    np.concatenate(probs_all),
        'valence':  np.concatenate(val_all),
        'arousal':  np.concatenate(aro_all),
        'is_valid': np.array(valid_mask),
    }

from tqdm import tqdm
OUT_MTDD = Path(WORK) / 'mtddamfn_predictions'
OUT_MTDD.mkdir(parents=True, exist_ok=True)

for fid in tqdm(COVERED_FILE_IDS):
    out_path = OUT_MTDD / f'{fid}.npz'
    if out_path.exists():
        if is_valid_npz(out_path):
            continue
        print(f"Corrupt {out_path.name} from a prior killed run; removing and re-extracting.")
        out_path.unlink()
    try:
        save_atomic(out_path, **extract_mtddamfn(fid))
    except Exception as e:
        print(f"FAIL {fid}: {e}")


## Cell 10 — HSEmotion extraction (enet_b0_8_va_mtl)


In [ ]:
from hsemotion.facial_emotions import HSEmotionRecognizer
from tqdm import tqdm

recog = HSEmotionRecognizer(model_name='enet_b0_8_va_mtl', device='cuda')

OUT_HSE = Path(WORK) / 'hsemotion_predictions'
OUT_HSE.mkdir(parents=True, exist_ok=True)

@torch.no_grad()
def extract_hsemotion(fid, batch_size=128*BATCH_MULT, crop_size=224):
    crops, valid_mask = [], []
    for i, crop, valid in iter_face_crops(fid, crop_size=crop_size):
        valid_mask.append(valid)
        crops.append(crop if valid else np.zeros((crop_size, crop_size, 3), dtype=np.uint8))

    probs_all, val_all, aro_all = [], [], []
    for start in range(0, len(crops), batch_size):
        chunk = crops[start:start+batch_size]
        out = recog.predict_emotions(chunk, logits=False)
        arr = np.asarray(out[1]) if isinstance(out, tuple) else np.asarray(out)
        # Expected shape (B, 10) = [8 probs, valence, arousal]
        probs_all.append(arr[:, :8])
        val_all.append(arr[:, 8])
        aro_all.append(arr[:, 9])

    return {
        'probs':    np.concatenate(probs_all),
        'valence':  np.concatenate(val_all),
        'arousal':  np.concatenate(aro_all),
        'is_valid': np.array(valid_mask),
    }

for fid in tqdm(COVERED_FILE_IDS):
    out_path = OUT_HSE / f'{fid}.npz'
    if out_path.exists():
        if is_valid_npz(out_path):
            continue
        print(f"Corrupt {out_path.name} from a prior killed run; removing and re-extracting.")
        out_path.unlink()
    try:
        save_atomic(out_path, **extract_hsemotion(fid))
    except Exception as e:
        print(f"FAIL {fid}: {e}")


## Cell 11 — EmoNet extraction


In [ ]:
import sys
sys.path.insert(0, '/content/emonet')
from emonet.models import EmoNet

n_classes = 8
emonet = EmoNet(n_expression=n_classes).to('cuda').eval()
ckpt = torch.load(f'/content/emonet/pretrained/emonet_{n_classes}.pth', map_location='cuda')
ckpt = {k.replace('module.', ''): v for k, v in ckpt.items()}
emonet.load_state_dict(ckpt, strict=False)

OUT_EMO = Path(WORK) / 'emonet_predictions'
OUT_EMO.mkdir(parents=True, exist_ok=True)

@torch.no_grad()
def extract_emonet(fid, batch_size=64*BATCH_MULT, crop_size=256):
    probs_all, val_all, aro_all, valid_mask = [], [], [], []
    batch_buf = []

    def flush():
        x = torch.from_numpy(np.stack(batch_buf)).permute(0,3,1,2).float().to('cuda') / 255.
        out = emonet(x)
        probs_all.append(torch.softmax(out['expression'], dim=-1).cpu().numpy())
        v = out['valence']; a = out['arousal']
        val_all.append(v.cpu().numpy().squeeze(-1) if v.dim()>1 else v.cpu().numpy())
        aro_all.append(a.cpu().numpy().squeeze(-1) if a.dim()>1 else a.cpu().numpy())
        batch_buf.clear()

    for i, crop, valid in iter_face_crops(fid, crop_size=crop_size):
        valid_mask.append(valid)
        batch_buf.append(crop if valid else np.zeros((crop_size, crop_size, 3), dtype=np.uint8))
        if len(batch_buf) == batch_size:
            flush()
    if batch_buf: flush()

    return {
        'probs':    np.concatenate(probs_all),
        'valence':  np.concatenate(val_all),
        'arousal':  np.concatenate(aro_all),
        'is_valid': np.array(valid_mask),
    }

for fid in tqdm(COVERED_FILE_IDS):
    out_path = OUT_EMO / f'{fid}.npz'
    if out_path.exists():
        if is_valid_npz(out_path):
            continue
        print(f"Corrupt {out_path.name} from a prior killed run; removing and re-extracting.")
        out_path.unlink()
    try:
        save_atomic(out_path, **extract_emonet(fid))
    except Exception as e:
        print(f"FAIL {fid}: {e}")


## Cell 12 — Evaluation: CCC + classification agreement vs Imitator


In [ ]:
def ccc(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = x[mask], y[mask]
    if len(x) < 2: return np.nan
    mx, my = x.mean(), y.mean()
    vx, vy = x.var(), y.var()
    cov = np.mean((x - mx) * (y - my))
    return 2 * cov / (vx + vy + (mx - my)**2 + 1e-12)

def pearson(x, y):
    x, y = np.asarray(x, float), np.asarray(y, float)
    mask = ~(np.isnan(x) | np.isnan(y))
    return np.corrcoef(x[mask], y[mask])[0,1] if mask.sum() > 1 else np.nan

def eval_one(fid, pred_dir):
    pred_path = Path(pred_dir) / f'{fid}.npz'
    if not pred_path.exists(): return None
    pred = np.load(pred_path)
    gt = load_imitator_gt(fid)
    if gt is None: return None

    n = min(len(pred['valence']), len(gt['valence']))
    mask = pred['is_valid'][:n].astype(bool)

    v_ccc = ccc(pred['valence'][:n][mask], gt['valence'][:n][mask])
    a_ccc = ccc(pred['arousal'][:n][mask], gt['arousal'][:n][mask])
    v_pearson = pearson(pred['valence'][:n][mask], gt['valence'][:n][mask])
    a_pearson = pearson(pred['arousal'][:n][mask], gt['arousal'][:n][mask])

    pred_cls = pred['probs'][:n][mask].argmax(-1)
    gt_cls   = gt['scores'][:n][mask].argmax(-1)
    cls_agree = (pred_cls == gt_cls).mean()

    return dict(file_id=fid, valence_ccc=v_ccc, arousal_ccc=a_ccc,
                valence_pearson=v_pearson, arousal_pearson=a_pearson,
                cls_agree=cls_agree, n=mask.sum())

rows = []
for model_name, pred_dir in [('MT-DDAMFN', OUT_MTDD), ('HSEmotion', OUT_HSE), ('EmoNet', OUT_EMO)]:
    for fid in COVERED_FILE_IDS:
        r = eval_one(fid, pred_dir)
        if r: rows.append({'model': model_name, **r})

results = pd.DataFrame(rows)
summary = results.groupby('model').agg(
    valence_ccc=('valence_ccc', 'mean'),
    arousal_ccc=('arousal_ccc', 'mean'),
    valence_pearson=('valence_pearson', 'mean'),
    arousal_pearson=('arousal_pearson', 'mean'),
    cls_agree=('cls_agree', 'mean'),
    n_files=('file_id', 'count'),
).round(3)
print(summary)

results.to_csv(f'{WORK}/bakeoff_per_file.csv', index=False)
summary.to_csv(f'{WORK}/bakeoff_summary.csv')


## Cell 13 — Visualize + label-ordering sanity check


In [ ]:
import matplotlib.pyplot as plt, seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
sns.boxplot(data=results, x='model', y='valence_ccc', ax=axes[0]); axes[0].set_title('Valence CCC vs Imitator')
sns.boxplot(data=results, x='model', y='arousal_ccc', ax=axes[1]); axes[1].set_title('Arousal CCC vs Imitator')
sns.boxplot(data=results, x='model', y='cls_agree',   ax=axes[2]); axes[2].set_title('8-class argmax agreement')
for a in axes: a.set_ylim(-0.1, 1.0)
plt.tight_layout(); plt.savefig(f'{WORK}/bakeoff_summary.png', dpi=150)
plt.show()

# Imitator's class ordering may not match AffectNet — check via confusion matrix
from sklearn.metrics import confusion_matrix
best_model = summary['cls_agree'].idxmax()
pred_dir = {'MT-DDAMFN': OUT_MTDD, 'HSEmotion': OUT_HSE, 'EmoNet': OUT_EMO}[best_model]
preds_all, gts_all = [], []
for fid in COVERED_FILE_IDS[:20]:
    pp = Path(pred_dir)/f'{fid}.npz'
    if not pp.exists(): continue
    pred = np.load(pp); gt = load_imitator_gt(fid)
    if gt is None: continue
    n = min(len(pred['valence']), len(gt['valence']))
    m = pred['is_valid'][:n].astype(bool)
    preds_all.append(pred['probs'][:n][m].argmax(-1))
    gts_all.append(gt['scores'][:n][m].argmax(-1))
cm = confusion_matrix(np.concatenate(gts_all), np.concatenate(preds_all))
plt.figure(figsize=(6,5))
sns.heatmap(cm / cm.sum(1, keepdims=True), annot=True, fmt='.2f',
            xticklabels=['An','Co','Di','Fe','Ha','Ne','Sa','Su'],
            yticklabels=['An','Co','Di','Fe','Ha','Ne','Sa','Su'])
plt.xlabel(f'{best_model} argmax'); plt.ylabel('Imitator argmax')
plt.title('Label-ordering check')
plt.tight_layout(); plt.savefig(f'{WORK}/label_ordering_check.png', dpi=150); plt.show()




If the confusion matrix is NOT diagonal-dominant, Imitator's class ordering differs from AffectNet's — find the permutation that maximizes the diagonal and re-run `cls_agree` with the permuted argmax.


## Cell 14 — Persist results back to csci535-project (optional)

If you want the bake-off CSVs + plots committed to the csci535-project repo:


In [ ]:
import shutil
TARGET = CSCI535_REPO / 'scripts' / 'valence_arousal' / 'bakeoff_results'
TARGET.mkdir(parents=True, exist_ok=True)
for name in ['bakeoff_per_file.csv', 'bakeoff_summary.csv', 'bakeoff_summary.png', 'label_ordering_check.png']:
    src = Path(WORK) / name
    if src.exists():
        shutil.copy2(src, TARGET / name)
print(f"Copied to {TARGET}")
# Since the repo lives in Drive, `git pull` on your local machine will NOT pick these up until
# you push from Colab (or copy them out of Drive). To push from Colab:
#   !cd "{CSCI535_REPO}" && git add scripts/valence_arousal/bakeoff_results && \
#     git -c user.name="rasikaramanan" -c user.email="sonaram14159@gmail.com" \
#         commit -m "Add bake-off results from Colab" && git push




Alternative: copy to Google Drive by uncommenting the Drive mount block in Cell 4 and setting `WORK` to a Drive path before running extraction.


## Cell 15 — Scale-up plan for all 443 interactions


In [ ]:
winner = summary.sort_values(['valence_ccc', 'arousal_ccc'], ascending=False).index[0]
print(f"Winner: {winner}")

bake_stats = pd.read_csv(f'{WORK}/bakeoff_per_file.csv')
avg_frames_per_file = bake_stats[bake_stats.model == winner]['n'].mean()
total_frames_443 = 443 * 2 * avg_frames_per_file
print(f"Avg frames/file: {avg_frames_per_file:.0f}")
print(f"Est total frames for 443: {total_frames_443:,.0f}")

# If the bake-off took W wall-seconds for 136 files, linear-scale projection:
#   full_runtime_h = W / 136 * 886 / 3600
# (fill in after bake-off completes)




**Optimization levers for the full 443 run:**
1. **torch.compile + float16.** Wrap the winning model in `torch.compile(model)` and cast to `float16`. 1.5-2× speedup on T4.
2. **Streaming + background download.** Pre-download the next interaction while the GPU is busy on the current one. Overlaps S3 latency with compute.
3. **Delete MP4/combined-NPZ after prediction** to stay under the 150 GB Colab VM disk quota. Only the prediction NPZs need to live long-term.
4. **Checkpoint to Drive every ~10 files.** Colab Pro disconnects at 12 h and can preempt earlier — resumability lets you restart without losing work.
5. **Shard across 2-3 parallel Colab sessions** by interaction ID mod N. 280 compute units is plenty.
6. **Filter by `is_valid_box`** to skip frames Seamless already marked invalid — saves ~5-15% per video.
7. **ONNX Runtime for HSEmotion** if that wins — ~1.3× faster than PyTorch on T4.

**Storage planning:** Each prediction NPZ is ~80-150 KB (n_frames × 10 × float32, compressed). 886 files × 120 KB ≈ 100 MB total — trivial to keep on Drive or push to the repo.


## Cell 16 — (After picking a winner) full extraction driver

Once the winner is known, duplicate the winning cell's extraction loop but iterate over all 886 participant-files (all 443 interactions × 2). Re-use the same `iter_face_crops` helper. Add the resumability + deletion logic from Cell 15's optimization list.


## Common issues + fixes

- **Cell 1 git clone fails with 401 / bad credentials:** the hardcoded PAT has expired or been revoked. Regenerate a fine-grained token with `Contents: Read` on `USC-CSCI-535-SP26/csci535-project` at github.com/settings/tokens and replace `GH_PAT` in Cell 1.
- **Cell 1 git clone is extremely slow:** expected on first run — you're writing thousands of git objects through the Drive FUSE mount. 8-15 min is normal; on subsequent runs the clone is skipped (just a `git pull`).
- **Cell 1 clone produces a repo with `.git/` but no worktree files:** the Drive mount silently dropped writes. Delete the repo folder in Drive via the file browser and re-run Cell 1.
- **Cell 3 shows 108 of 136 NPZs even after re-cloning:** the 28 `emotion_features.npz` files under `single_annotated_interactions/` may be committed locally but not pushed to GitHub. From your local macOS checkout run `git status`, then `git push` if there are unpushed commits, and run `!cd "{CSCI535_REPO}" && git pull` in Colab. (If `git status` shows the NPZs as untracked, `git add -f` them first — check `git check-ignore -v ...` to verify they aren't being gitignored.)
- **Cell 5 raises `ModuleNotFoundError: No module named 'seamless_interaction'`:** the Cell 2 editable install silently failed (common when `pip install -e .` runs against a FUSE-mounted Drive path, or when the kernel was restarted after install). Cell 4 now inserts `{SEAMLESS_REPO}/src` into `sys.path` as a belt-and-suspenders fix — make sure you've re-run Cell 4 after any kernel restart. If the error persists, verify the package source is where expected: `!ls "{SEAMLESS_REPO}/src/seamless_interaction/fs.py"`.
- **Cell 6 downloads fail for some files:** the label/split in filelist.csv for that file may be incorrect. The script prints which ones failed — re-try with the correct `DatasetConfig(label=..., split=...)`.
- **Cell 9 reports many missing keys:** MT-DDAMFN weight file name or state_dict structure differs from the template. `ls /content/fer_savchenko/src/ABAW/ABAW6/models/` and update `MT_DDAMFN_WEIGHTS`. Print `sd.keys()` to see the tensor layout.
- **OOM on EmoNet (Cell 11) or any extraction cell:** lower `BATCH_MULT` in Cell 4 — e.g. if Cell 4 set it to 6 and Cell 11 OOMs, set `BATCH_MULT = 2` and re-run from Cell 11. EmoNet is the most memory-hungry (256² crops + 70M params); it's usually the first to OOM when scaling up.
- **"CUDA error: no kernel image is available for execution on this device" in any model cell:** the installed PyTorch lacks kernels for the GPU architecture Colab gave you (most common on brand-new GPUs like Blackwell / sm_120). Run the nightly-install command Cell 4 prints when its smoke test fails, then restart the runtime and re-run from Cell 2.
- **Imitator scores argmax always returns the same class:** the raw-logit ordering may differ from AffectNet. Check Cell 13's confusion matrix and apply the right permutation before computing cls_agree.
